# Training batch-size probe
Use standard CPU first; verify the previously prepared 42M finalist and original training pool. Then choose a BF16-capable GPU with at least 40 GiB. Eight short throughput measurements use equal examples; all probe updates are discarded. No full training or bot submission. Do not Run All.


In [ ]:
from pathlib import Path
import json,hashlib
assert not Path('/dev/nvidia0').exists(), 'CPU preparation first'
from google.colab import drive
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
p=root/'prepared/production-batch-finalist6k-v1'
gate=json.loads((p/'ready.json').read_text())
inputs=root/'prepared/inference-finalist6k-v1'
ready=json.loads((inputs/'ready.json').read_text())
assert gate['cpu_verified'] and ready['cpu_preflight_passed']
assert gate['model_sha256']==ready['model_sha256']
assert hashlib.sha256((p/'probe.jsonl').read_bytes()).hexdigest()==gate['pool_sha256']
assert hashlib.sha256((inputs/'inputs.tar').read_bytes()).hexdigest()==ready['inputs_sha256']
print('CPU_BATCH_INPUTS_VERIFIED',gate)


Only after CPU verification, switch to the training GPU.


In [ ]:
from google.colab import drive,runtime
try:
    drive.mount('/content/drive',timeout_ms=90000)
    from pathlib import Path
    import os,sys,json,hashlib,subprocess,tarfile,shutil,asyncio,signal,time
    os.environ['USE_TF']='0';os.environ['USE_FLAX']='0'
    root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
    prepared=root/'prepared/production-batch-finalist6k-v1'
    gate=json.loads((prepared/'ready.json').read_text())
    inputs=root/'prepared/inference-finalist6k-v1'
    ready=json.loads((inputs/'ready.json').read_text())
    assert gate['cpu_verified'] and ready['cpu_preflight_passed']
    assert gate['model_sha256']==ready['model_sha256']
    work=Path('/content/hw2-production-batch-probe');work.mkdir(exist_ok=False)
    bundle=work/'inputs.tar';shutil.copyfile(inputs/'inputs.tar',bundle)
    assert hashlib.sha256(bundle.read_bytes()).hexdigest()==ready['inputs_sha256']
    with tarfile.open(bundle) as z:z.extractall(work,filter='data')
    shutil.copyfile(prepared/'probe.jsonl',work/'probe.jsonl')
    assert hashlib.sha256((work/'probe.jsonl').read_bytes()).hexdigest()==gate['pool_sha256']
    os.chdir(work)
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6'],check=True,timeout=180)
    import torch
    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    assert torch.cuda.get_device_properties(0).total_memory>=40*1024**3
    output=root/'benchmarks/production-batch-finalist6k-g4-v1'
    assert not output.exists()
    pilot_log=(prepared/'execution.log').open('w')
    command=[sys.executable,'-u','-m','homeworks.hw2.benchmark_production_batches','--checkpoint','model','--pool','probe.jsonl','--output',str(output),'--gate',str(prepared/'ready.json')]
    pilot_process=subprocess.Popen(command,stdout=pilot_log,stderr=subprocess.STDOUT,start_new_session=True)
    async def finish_probe():
        status={'pid':pilot_process.pid,'started_utc':time.time(),'source_commit':ready['source_commit']}
        try:
            await asyncio.wait_for(asyncio.to_thread(pilot_process.wait),timeout=600)
            status['returncode']=pilot_process.returncode
        except BaseException as error:
            status['error']=type(error).__name__+': '+str(error)
            if pilot_process.poll() is None:
                os.killpg(pilot_process.pid,signal.SIGTERM)
                try:await asyncio.wait_for(asyncio.to_thread(pilot_process.wait),timeout=10)
                except asyncio.TimeoutError:os.killpg(pilot_process.pid,signal.SIGKILL)
        finally:
            pilot_log.close();status['finished_utc']=time.time()
            try:(prepared/'execution.complete.json').write_text(json.dumps(status,indent=2))
            finally:runtime.unassign()
    pilot_task=asyncio.create_task(finish_probe())
    print('BATCH_PROBE_STARTED',pilot_process.pid)
except BaseException:
    runtime.unassign()
    raise


Verify the runtime actually disconnects after completion; manually release if needed. Inspect Drive benchmark results on CPU. A faster batch needs an equal-example learning comparison before adoption.
